In [1]:
%env DATA_PATH=../../../data
from lib.fit import load_fit_file, get_gps_data, get_camera_starts, get_camera_ends, get_sensor_data
import json
import pandas as pd
import os
from tqdm.notebook import tqdm
import plotly.express as px
import subprocess

DATA_PATH = '../../../data'

env: DATA_PATH=../../../data


In [2]:
fit = load_fit_file(f"{DATA_PATH}/archive/tmp/calibrate/2026-05-09-16-54-00.fit")

In [3]:
# s = stabalization, lc = lens correction
# idx_no_s = 2
# idx_no_s_lc = 3
# idx = idx_no_s_lc
idx = 0
camera_starts = get_camera_starts(fit)
camera_ends = get_camera_ends(fit)
start, end = camera_starts[idx], camera_ends[idx]
bag_folder = f"{DATA_PATH}/archive/tmp/calibrate/large_data{idx}"
os.makedirs(bag_folder, exist_ok=True)

## IMU

In [4]:
calibration_mesgs = fit['three_d_sensor_calibration_mesgs']
calibration_data = { m['sensor_type']: m for m in calibration_mesgs }

In [5]:
accel_cal = calibration_data['accelerometer']
accel_raw, accel_data, fs = get_sensor_data(accel_cal, fit['accelerometer_data_mesgs'], {'alpha_x': 'accel_x', 'alpha_y': 'accel_y', 'alpha_z': 'accel_z'})
accel = accel_data.loc[start:end].drop(columns=['timestamp'])
accel

,alpha_x,alpha_y,alpha_z
timestamp,,,
25964,-0.051270,0.038574,-0.937012
25974,-0.041504,0.032227,-0.933105
25984,-0.026855,0.041016,-0.951172
25994,-0.021484,0.055176,-0.966309
26004,0.004883,0.059082,-0.980469
...,...,...,...
146932,-0.020996,0.075195,-0.989746
146942,-0.042969,0.083008,-0.996582
146952,-0.039062,0.088379,-0.995117


In [6]:
gyro_cal = calibration_data['gyroscope']
gyro_raw, gyro_data, gyro_fs = get_sensor_data(gyro_cal, fit['gyroscope_data_mesgs'], {'omega_x': 'gyro_x', 'omega_y': 'gyro_y', 'omega_z': 'gyro_z'})
gyro = gyro_data.loc[start:end].drop(columns=['timestamp'])
gyro

,omega_x,omega_y,omega_z
timestamp,,,
25964,-4.390244,4.085366,-0.060976
25974,-6.585366,2.865854,0.304878
25984,-7.317073,-0.914634,-1.036585
25994,-6.707317,-3.902439,-2.743902
26004,-4.634146,-6.341463,-5.243902
...,...,...,...
146932,0.426829,-22.256098,-9.268293
146942,-2.195122,-19.024390,-12.621951
146952,-2.926829,-13.353659,-13.841463


In [8]:
gyro_mag = (gyro**2).sum(axis=1)**0.5
px.line(gyro_mag)

In [7]:
imu = pd.merge_asof(accel, gyro, on='timestamp')
imu.timestamp = (imu.timestamp * 1e6).astype('int64')
imu.to_csv(f"{bag_folder}/imu0.csv", index=False, header=True)

In [6]:
with open(f"{DATA_PATH}/archive/tmp/calibrate/fit.json", 'w') as f:
    json.dump(fit, f, default=str, indent=2)

## Video

In [9]:
import cv2
# videos = ['VIRB0119.MP4', 'VIRB0120.MP4', 'VIRB0121.MP4', 'VIRB0122.MP4']
videos = ['VIRB0129.MP4']
p = f"{DATA_PATH}/archive/tmp/calibrate/{videos[idx]}"

In [13]:
cap = cv2.VideoCapture(p)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {p}")

frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
progress = tqdm(total=frame_count, desc="Saving frames")

saved_frames = 0
os.makedirs(os.path.join(bag_folder, "cam0"), exist_ok=True)
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        timestamp_ms = start + cap.get(cv2.CAP_PROP_POS_MSEC)
        timestamp_ns = int(round(timestamp_ms * 1_000_000))
        out_path = os.path.join(bag_folder, f"cam0/{timestamp_ns}.png")
        
        cv2.imwrite(out_path, frame)
        saved_frames += 1
        progress.update(1)
finally:
    progress.close()
    cap.release()

saved_frames

Saving frames:   0%|          | 0/7261 [00:00<?, ?it/s]

7261

In [50]:
((end - start) / 1000) * 60

7558.5

## IMU Noise

In [2]:
fit = load_fit_file(f"{DATA_PATH}/archive/tmp/calibrate/2026-05-07-19-44-00.fit")

Caching archive_tmp_calibrate_2026-05-07-19-44-00.json


In [23]:
calibration_mesgs = fit['three_d_sensor_calibration_mesgs']
calibration_data = { m['sensor_type']: m for m in calibration_mesgs }
start, end = 4_000_000, 15_000_000

In [17]:
accel_cal = calibration_data['accelerometer']
accel_raw, accel_data, fs = get_sensor_data(accel_cal, fit['accelerometer_data_mesgs'], {'alpha_x': 'accel_x', 'alpha_y': 'accel_y', 'alpha_z': 'accel_z'})
accel = accel_data.drop(columns=['timestamp']).loc[start:end]
accel

,alpha_x,alpha_y,alpha_z
timestamp,,,
4000006,0.080078,0.935547,-0.047363
4000016,0.080566,0.934570,-0.047852
4000026,0.081543,0.935059,-0.046875
4000036,0.082031,0.937500,-0.045898
4000046,0.081543,0.938965,-0.045898
...,...,...,...
14999957,0.081543,0.935547,-0.046387
14999967,0.082520,0.935547,-0.046387
14999977,0.082031,0.936523,-0.046387


In [18]:
gyro_cal = calibration_data['gyroscope']
gyro_raw, gyro_data, gyro_fs = get_sensor_data(gyro_cal, fit['gyroscope_data_mesgs'], {'omega_x': 'gyro_x', 'omega_y': 'gyro_y', 'omega_z': 'gyro_z'})
gyro = gyro_data.drop(columns=['timestamp']).loc[start:end]
gyro

,omega_x,omega_y,omega_z
timestamp,,,
4000006,-0.121951,-0.975610,0.731707
4000016,-0.060976,-0.975610,0.731707
4000026,-0.060976,-0.914634,0.731707
4000036,-0.060976,-0.975610,0.792683
4000046,-0.121951,-0.914634,0.853659
...,...,...,...
14999957,0.000000,-0.853659,0.853659
14999967,0.000000,-0.914634,0.853659
14999977,0.060976,-0.975610,0.853659


In [20]:
imu = pd.merge_asof(accel, gyro, on='timestamp')
imu.timestamp = (imu.timestamp * 1e6).astype('int64')
imu.to_csv(f"{DATA_PATH}/archive/tmp/calibrate/noise_bag/imu0.csv", index=False, header=True)

In [24]:
(end - start) / 1000

11000.0